# Experiments with the Text Charter

## Imports

In [ ]:
import getpass
import os

from langchain_openai import OpenAIEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

import openai

Set up API key

In [ ]:
os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API key: ')
print("OpenAI version:", openai.__version__)
client = openai.OpenAI()  # uses the OPENAI_API_KEY from env var


Define the charter


In [ ]:
CHARTER_RULES = [
    "Do not harm humans or cause fear.",
    "Offer assistance if an agent is in need or requests help.",
    "Always maintain a safe distance from humans unless invited to approach.",
    "Never block other agents who want to pass.",
    "Comply with social norms (e.g., don't interrupt conversations)."
]

def violates_charter(action: str) -> list:
    violations = []
    if "push" in action.lower():
        violations.append("Do not harm humans or cause fear.")
    if "ignore" in action.lower():
        violations.append("Offer assistance if an agent is in need.")
    return violations



✅ Case Repository

In [ ]:
cases = [
    "Title: Guiding Lost Person\nContext: A person is lost in the hallway.\nRobot helps them find their way.",
    "Title: Yielding Right of Way\nContext: Human carrying box.\nRobot steps aside politely.",
    "Title: Blocking Human\nContext: Robot blocks door.\nOutcome: Negative. Human was frustrated."
]

embeddings = OpenAIEmbeddings()
vectordb = FAISS.from_texts(cases, embeddings)

Textual Simulation Environment

In [ ]:
class TextWorld:
    def __init__(self):
        self.state = {
            "RobotA": {"location": "Hallway", "status": "idle"},
            "HumanB": {"location": "Hallway", "status": "lost"},
            "Door": "closed"
        }

    def describe(self):
        return f"RobotA is in {self.state['RobotA']['location']}.\nHumanB is {self.state['HumanB']['status']}.\nDoor is {self.state['Door']}."

    def apply_action(self, action):
        if "approach" in action.lower() and "HumanB" in action:
            self.state["HumanB"]["status"] = "being helped"
            return "RobotA approached and offered help. HumanB accepted."
        elif "open door" in action.lower():
            self.state["Door"] = "open"
            return "RobotA opened the door."
        return "Action unclear or not implemented."


Strategizer (LLM Planner)


In [ ]:

def build_prompt(world, retrieved, charter):
    return f"""You are a prosocial robot. Follow these rules:\n{charter}\n
WORLD STATE:\n{world}\n
PAST CASES:\n{retrieved}\n
What should RobotA do next?\nGive your reasoning. Then write: ACTION: <do something>
"""

def get_plan(prompt):
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content


Run Scenario

In [ ]:
env = TextWorld()
description = env.describe()
query = "help a lost person"
retrieved = vectordb.similarity_search(query, k=2)
retrieved_text = "\n---\n".join([doc.page_content for doc in retrieved])

prompt = build_prompt(description, retrieved_text, CHARTER_RULES)
response = get_plan(prompt)
print("\n\n🧠 LLM Response:\n", response)

action_line = [line for line in response.split("\n") if line.startswith("ACTION:")][0]
action = action_line.replace("ACTION:", "").strip()

violations = violates_charter(action)
if violations:
    print(f"\n⚠️ Charter Violations: {violations}")
else:
    print("\n✅ Action approved by Charter.")
    result = env.apply_action(action)
    print("Result:", result)

